# Предобработка данных

### Таблица с оценками

In [1]:
import pandas as pd  
import numpy as np  
import re

In [2]:
CAMPUS = 'НИУ ВШЭ - Санкт-Петербург'
FACULTY = ['факультет "Школа информатики, физики и технологий"', 'факультет Санкт-Петербургская школа физико-математических и компьютерных наук']

In [3]:
grades = pd.read_csv('grades.csv', sep=';', low_memory=False)

- Нас интересует только факультет ШИФТ СПб

In [4]:
grades = grades[ grades['campus'] == CAMPUS ]
grades = grades[ grades['faculty'].isin(FACULTY) ]

In [5]:
grades['student_id_hash'].value_counts()

student_id_hash
cee6d1501f1b1c8bc2be658556d18d4e    84
de1c077ca82ab440c2d284793fb1efd4    79
752de4f1450be612cec55d16c89f77df    78
07d950d6f8a3da8c9231031cf6f37aea    78
c21eec00a9757200e0ddb1fc27e8a60e    77
                                    ..
185a7a1774fdadb25e50b2ea24f37c84     1
2b538134f0580a4a96d90b9c8899c54d     1
b27019c3264bec527047134a1d0d378e     1
526402f427e868298969443bcf14ab0e     1
6e627180145bf2f6417bba8b3793d289     1
Name: count, Length: 571, dtype: int64

In [6]:
grades.columns

Index(['student_id_hash', 'campus', 'faculty', 'program', 'education_level',
       'course', 'group', 'place_type', 'subject_name', 'exam_type',
       'subject_unit', 'grade_10', 'absence_status', 'module', 'academic_year',
       'student_status'],
      dtype='str')

- Удаляем ненужные колонки, так как кампус и факультет выбраны

In [7]:
grades = grades.drop(columns=['campus', 'faculty', 'subject_unit'])

- Отдельно сохраняем информацию о студентах

In [8]:
students_col = ['student_id_hash', 'program', 'education_level', 'place_type']

students = grades[students_col].drop_duplicates(subset=['student_id_hash']).reset_index(drop=True)

In [9]:
students['student_id'] = range(1, len(students) + 1)
id_map = dict(zip(students['student_id_hash'], students['student_id']))

In [10]:
students = students.set_index('student_id')
students.head()

,student_id_hash,program,education_level,place_type
student_id,,,,
1,6034a1f41662a6e36c1a723243dbebc2,Прикладная математика и информатика,Бакалавриат,Бюджетные
2,590e486262f8d6535aaa9b9778c8586f,Прикладной анализ данных и искусственный интел...,Бакалавриат,Коммерческие за счет средств вуза
3,af1efce0752ddd23d4f47d0ecfa75d48,Прикладная математика и информатика,Бакалавриат,Бюджетные
4,51fa6fd4677884c4da2480982bb94c6e,Прикладной анализ данных и искусственный интел...,Бакалавриат,Бюджетные
5,dadf51709e1817d350306f3e8c03680c,Прикладной анализ данных и искусственный интел...,Бакалавриат,Коммерческие за счет средств вуза


- Отдельно сохраняем информацию об оценках студентов

In [11]:
data_col = ['student_id_hash', 'course', 'subject_name', 'exam_type', 'grade_10', 'absence_status','module', 'academic_year']
data = grades[data_col].copy()
data['student_id'] = data['student_id_hash'].map(id_map)

In [12]:
def get_discipline_type_and_name(line):
    tp, name = line.split(': ', 1)
    return pd.Series([tp, name])

In [13]:
data[['class_type', 'discipline']] = data['subject_name'].apply(get_discipline_type_and_name)

data = data[data['class_type'] == 'Дисциплина']
data = data.drop(columns=['class_type', 'subject_name'])

In [14]:
students = students.drop(columns=['student_id_hash'])
data = data.drop(columns=['student_id_hash'])

In [15]:
data_columns = ['student_id', 'course', 'exam_type', 'grade_10', 'absence_status','module', 'academic_year', 'discipline']
data = data[data_columns].copy()
data.head()

,student_id,course,exam_type,grade_10,absence_status,module,academic_year,discipline
63873,1,4,Первая сдача,9,\N,1,2024,Веб-поиск и ранжирование
63896,2,4,Первая сдача,6,\N,1,2024,Теория вероятностей и математическая статистика
63909,3,3,Первая сдача,7,\N,1,2024,Формальные языки
63927,4,3,Первая сдача,10,\N,1,2024,Дискретная математика
63930,5,4,Первая сдача,7,\N,1,2024,Теория вероятностей и математическая статистика


- Преобразуем числовые значения в тип _int_

In [16]:
data['course'] = data['course'].astype("int8")
data['module'] = data['module'].astype("int8")
data['academic_year'] = data['academic_year'].astype("int32")

data = data[ data['grade_10'] != '\\N' ]
data['grade_10'] = data['grade_10'].astype("int8")

In [17]:
data.dtypes

student_id        int64
course             int8
exam_type           str
grade_10           int8
absence_status      str
module             int8
academic_year     int32
discipline          str
dtype: object

**Итог:**

- Исходная таблица `grades`
- Список всех студентов ( ключ \ индекс $-$ уникальный id студента)
- Список оценок студентов (для одного и того же студента может быть несколько записей о различных оценках за разный период, по разным предметам и тп)

### Таблица СОП (студенческая оценка преподавания)

In [18]:
sop = pd.read_csv('sop.csv', sep=';', low_memory=False)

In [19]:
sop.dtypes

period                       str
branch                       str
faculty                      str
level                        str
program                      str
course                     int64
discipline                   str
discipline_key               str
discipline_type              str
discipline_branch            str
discipline_faculty           str
discipline_department        str
class_type                   str
question                     str
teacher_name_hash            str
students_enrolled          int64
students_responded         int64
students_unsure            int64
avg_score                float64
std_deviation            float64
score_5                  float64
score_4                  float64
score_3                  float64
score_2                  float64
score_1                  float64
dtype: object

- Рассматриваем только ШИФТ СПб
- Для нашего факультета все данные СОП даны за определенные модули, а не семестры

In [20]:
sop = sop[ sop['branch'] == CAMPUS ]
sop['faculty'] = sop['faculty'].replace('Факультет Санкт-Петербургская школа физико-математических и компьютерных наук', 'факультет Санкт-Петербургская школа физико-математических и компьютерных наук')
sop = sop[ sop['faculty'].isin(FACULTY) ]

c = [p for p in sop['period'].unique() if 'семестр' in p] 
c

[]

- Из всех вопросов в СОПе потенциально могут пригодиться только те, что связаны с оценкой сложности курса

In [21]:
QUESTIONS = ['Сложность курса для успешного прохождения («1» – курс очень легкий, «5» - курс очень сложный для прохождения)', 
             'Сложность данного онлайн-курса для успешного прохождения («1» – курс очень легкий, «5» - курс очень сложный для прохождения)', 
             'Сложность элемента практической подготовки для успешного завершения («1» – очень легко, «5» - очень сложно)',
             'Сложность практики для успешного завершения («1» – очень легко, «5» - очень сложно)',
             'Сложность проекта для успешного завершения («1» – проект очень легкий, «5» - проект очень сложный для прохождения)']

sop = sop[sop['question'].isin(QUESTIONS)]

In [22]:
sop['education_level'] = sop['level'].replace({'Б': 'Бакалавриат', 'М': 'Магистратура', 'С': 'Специалитет'})

In [23]:
def get_module_and_year(line):
    y1, y2 = int(line[:4]), int(line[5:9])
    m = int(re.search(r'(\d)', line[9:]).group(1))

    if m < 3:
        return pd.Series([m, y1])
    else:
        return pd.Series([m, y2])
    
    
sop[['module', 'academic_year']] = sop['period'].apply(get_module_and_year)

- Удаляем ненужные колонки и оставляем только дисциплины

In [24]:
sop = sop.drop(columns=['period', 'branch', 'faculty', 'level', 'discipline_key', 'discipline_branch',
       'discipline_faculty', 'discipline_department', 'discipline_type', 'teacher_name_hash', 'students_unsure', 'score_5', 'score_4',
       'score_3', 'score_2', 'score_1'])

sop = sop.query("class_type not in ['Элемент практической подготовки', 'Проектная деятельность',  'Практика']")

sop = sop.drop(columns='class_type')

- Преобразуем `course`, `module`, `academic_year` в те же типы, что и для таблицы с оценками

In [25]:
sop['module'] = sop['module'].astype("int8")
sop['course'] = sop['course'].astype("int8")
sop['academic_year'] = sop['academic_year'].astype("int32")

- Колонка с вопросом нам не понадобится, так как оставили только вопросы про сложность прохождения курса

In [26]:
sop = sop.drop(columns=['question'])

In [27]:
sop = sop.rename(columns={'avg_score': 'difficulty_avg_score'})

- Сделаем колонку с долей ответивших в СОПе студентов (вместо двух колонок `students_responded` и `students_enrolled`)

In [28]:
sop['students_responsed_ratio'] = sop['students_responded'] / sop['students_enrolled']
sop = sop.drop(columns=['students_enrolled', 'students_responded'])

In [29]:
sop.head()

,program,course,discipline,difficulty_avg_score,std_deviation,education_level,module,academic_year,students_responsed_ratio
82975,Б 03.03.02 ШИФТ СПБ 2022 очная Физика,1,Python для анализа данных,3.0000,1.08012,Бакалавриат,4,2025,0.857143
82982,М 01.04.02 ШИФТ СПБ 2020 очная Машинное обучен...,1,Web-поиск,4.3000,0.90000,Магистратура,4,2025,0.833333
82992,М 01.04.02 ШИФТ СПБ 2024 очная Проектирование ...,1,Администрирование и оптимизация производительн...,2.8750,0.78062,Магистратура,4,2025,0.941176
83005,Б 09.03.01 ШИФТ СПБ 2024 очная Компьютерные те...,1,Алгебра и геометрия,4.3636,0.64282,Бакалавриат,4,2025,0.846154
83012,Б 01.03.02 ШИФТ СПБ 2022 очная Прикладной анал...,1,Алгоритмы и структуры данных,3.7692,0.69657,Бакалавриат,4,2025,0.896552


- Еще одна _проблема:_ в таблицах по разному записаны названия программ

In [30]:
students['program'].unique()

<StringArray>
[                                'Прикладная математика и информатика',
                  'Прикладной анализ данных и искусственный интеллект',
                                    'Программирование и анализ данных',
                                                              'Физика',
                 'UX-аналитика и проектирование информационных систем',
                             'Компьютерные технологии, системы и сети',
 'Проектирование и разработка высоконагруженных информационных систем',
                                   'Машинное обучение и анализ данных',
                            'Вычислительная биология и биоинформатика',
           'Внедрение и оптимизация комплексных информационных систем']
Length: 10, dtype: str

In [31]:
sop['program'].unique()

<StringArray>
[                                                                  'Б 03.03.02 ШИФТ СПБ 2022 очная Физика',
                                        'М 01.04.02 ШИФТ СПБ 2020 очная Машинное обучение и анализ данных',
      'М 01.04.02 ШИФТ СПБ 2024 очная Проектирование и разработка высоконагруженных информационных систем',
                                  'Б 09.03.01 ШИФТ СПБ 2024 очная Компьютерные технологии, системы и сети',
                       'Б 01.03.02 ШИФТ СПБ 2022 очная Прикладной анализ данных и искусственный интеллект',
                                      'Б 01.03.02 ШИФТ СПБ 2020 очная Прикладная математика и информатика',
                                                                   'М 03.04.02 ШИФТ СПБ 2022 очная Физика',
                                 'М 01.04.02 ШИФТ СПБ 2022 очная Вычислительная биология и биоинформатика',
                      'М 01.04.02 ШИФТ СПБ 2022 очная UX-аналитика и проектирование информационных систем',
 'М 01.04.02 С

In [32]:
sop['program'] = sop['program'].str.extract(r'очная\s+([А-Я][А-Яа-я\s]*)')
sop = sop[ sop['program'].notna() ]
sorted(sop['program'].unique())

['Анализ данных в финансах',
 'Внедрение и оптимизация комплексных информационных систем',
 'Вычислительная биология и биоинформатика',
 'Информационные системы и взаимодействие человек',
 'Компьютерные технологии',
 'Машинное обучение и анализ данных',
 'Прикладная математика и информатика',
 'Прикладной анализ данных и искусственный интеллект',
 'Программирование и анализ данных',
 'Проектирование и разработка высоконагруженных информационных систем',
 'Теоретическая и математическая физика',
 'Физика']

In [33]:
sorted(students['program'].unique())

['UX-аналитика и проектирование информационных систем',
 'Внедрение и оптимизация комплексных информационных систем',
 'Вычислительная биология и биоинформатика',
 'Компьютерные технологии, системы и сети',
 'Машинное обучение и анализ данных',
 'Прикладная математика и информатика',
 'Прикладной анализ данных и искусственный интеллект',
 'Программирование и анализ данных',
 'Проектирование и разработка высоконагруженных информационных систем',
 'Физика']

In [34]:
data.to_csv('data.csv', index=False)
students.to_csv('students.csv', index=False)
sop.to_csv('sop_edited.csv', index=False)

**Итог:**

- Таблица с полезными для нас данными СОП: 
    - дисциплина, курс, год и модуль, сложность

### Подготовка объектов и признаков

In [35]:
objects = data.copy()

objects = objects.sort_values([
    'student_id', 
    'discipline', 
    'academic_year', 
    'module'
])

In [36]:
df = pd.merge(objects, students, how='left', on='student_id')

df = pd.merge(df, sop, how='left', on=['program', 'course', 'education_level', 'academic_year', 'module', 'discipline'])
df.head()

,student_id,course,exam_type,grade_10,absence_status,module,academic_year,discipline,program,education_level,place_type,difficulty_avg_score,std_deviation,students_responsed_ratio
0,1,4,Первая сдача,7,\N,2,2023,Альтернативные языки для JVM,Прикладная математика и информатика,Бакалавриат,Бюджетные,NaN,NaN,NaN
1,1,4,Первая сдача,9,\N,3,2024,Анализ программ,Прикладная математика и информатика,Бакалавриат,Бюджетные,4.0000,0.81650,0.750000
2,1,4,Первая сдача,8,\N,2,2023,Базы данных,Прикладная математика и информатика,Бакалавриат,Бюджетные,NaN,NaN,NaN
3,1,4,Первая сдача,9,\N,1,2024,Веб-поиск и ранжирование,Прикладная математика и информатика,Бакалавриат,Бюджетные,4.5000,0.80623,0.666667
4,1,4,Первая сдача,7,\N,2,2024,Веб-поиск и ранжирование,Прикладная математика и информатика,Бакалавриат,Бюджетные,4.0909,1.31111,0.733333


Лаги: 

- информация о среднем балле конкретного студента по всем дисциплинам за предыдущие модули (если данные имеются)
- минимальная полученная оценка за весь период обучения
- максимальная полученная оценка за весь период обучения
- предыдущие оценки по данной дисциплине, а также уточняющая информация (какая попытка экзамена, средняя сложность предмета в данном модуле и тп.)

In [37]:
total = df.copy()
total['grade_10'] = pd.to_numeric(total['grade_10'], errors='coerce')

- Определим порядок в столбце `exam_type` и отсортируем таблицу:

In [38]:
total['exam_type'].unique()

<StringArray>
[                     'Первая сдача',                         'Пересдача',
             'Пересдача с комиссией', 'Пересдача по уважительной причине']
Length: 4, dtype: str

In [39]:
total['exam_type'] = pd.Categorical(total['exam_type'], categories=['Первая сдача','Пересдача по уважительной причине','Пересдача','Пересдача с комиссией'], ordered=True)

total = total.sort_values(['student_id','academic_year','module','exam_type','discipline']).reset_index(drop=True)

Группируем по `student_id` для нахождения среднего балла, минимальной и максимальной оценки, а для добавления последних оценок группируем по `student_id` и `discipline`:

In [40]:
g_student = total.groupby('student_id')
g_subject = total.groupby(['student_id', 'discipline'])

total['grade_prev'] = g_subject['grade_10'].shift(1)

total['difficulty_prev'] = g_subject['difficulty_avg_score'].shift(1)

total['students_responsed_ratio_prev'] = g_subject['students_responsed_ratio'].shift(1)

total['std_deviation_prev'] = g_subject['std_deviation'].shift(1)

total['exam_type_prev'] = g_subject['exam_type'].shift(1)

In [41]:
total['cnt_prev'] = g_student.cumcount()

total['sum_prev'] = g_student['grade_10'].transform(lambda x: x.shift().cumsum())

total['min_prev'] = g_student['grade_10'].transform(lambda x: x.shift().cummin())

total['max_prev'] = g_student['grade_10'].transform(lambda x: x.shift().cummax())

total['avg_grade_prev'] = total['sum_prev'] / total['cnt_prev']

total = total[total['cnt_prev'] > 0].copy()

### Target

> Предсказываем следующую оценку и попытку сдачи

In [42]:
total['target_grade'] = g_subject['grade_10'].shift(-1)
total['target_type'] = g_subject['exam_type'].shift(-1)
total = total[~total['target_grade'].isna()].copy()


In [43]:
total.columns

Index(['student_id', 'course', 'exam_type', 'grade_10', 'absence_status',
       'module', 'academic_year', 'discipline', 'program', 'education_level',
       'place_type', 'difficulty_avg_score', 'std_deviation',
       'students_responsed_ratio', 'grade_prev', 'difficulty_prev',
       'students_responsed_ratio_prev', 'std_deviation_prev', 'exam_type_prev',
       'cnt_prev', 'sum_prev', 'min_prev', 'max_prev', 'avg_grade_prev',
       'target_grade', 'target_type'],
      dtype='str')

In [44]:
cols = [
    'student_id',
    'program', 'education_level', 'academic_year', 'place_type', 'course', 'absence_status', 'discipline',
    'module', 'exam_type', 'grade_10', 'difficulty_avg_score', 'std_deviation','students_responsed_ratio',
    'exam_type_prev','grade_prev', 'difficulty_prev', 'std_deviation_prev', 'students_responsed_ratio_prev',
    
    'avg_grade_prev',
    'min_prev','max_prev', 'target_grade', 'target_type'
    ]

total = total.reindex(columns=cols)
total.head()

,student_id,program,education_level,academic_year,place_type,course,absence_status,discipline,module,exam_type,...,exam_type_prev,grade_prev,difficulty_prev,std_deviation_prev,students_responsed_ratio_prev,avg_grade_prev,min_prev,max_prev,target_grade,target_type
13,1,Прикладная математика и информатика,Бакалавриат,2024,Бюджетные,4,\N,Веб-поиск и ранжирование,1,Первая сдача,...,NaN,NaN,NaN,NaN,NaN,7.384615,4.0,10.0,7.0,Первая сдача
17,1,Прикладная математика и информатика,Бакалавриат,2024,Бюджетные,4,\N,Правовая грамотность,2,Первая сдача,...,NaN,NaN,NaN,NaN,NaN,7.647059,4.0,10.0,8.0,Первая сдача
25,2,Прикладной анализ данных и искусственный интел...,Бакалавриат,2022,Коммерческие за счет средств вуза,4,\N,Дискретная математика,1,Первая сдача,...,NaN,NaN,NaN,NaN,NaN,7.000000,7.0,7.0,7.0,Первая сдача
27,2,Прикладной анализ данных и искусственный интел...,Бакалавриат,2022,Коммерческие за счет средств вуза,4,\N,Алгоритмы и структуры данных,2,Первая сдача,...,NaN,NaN,NaN,NaN,NaN,6.666667,6.0,7.0,7.0,Первая сдача
29,2,Прикладной анализ данных и искусственный интел...,Бакалавриат,2022,Коммерческие за счет средств вуза,4,\N,Дискретная математика,2,Первая сдача,...,Первая сдача,7.0,NaN,NaN,NaN,6.200000,5.0,7.0,7.0,Первая сдача


In [ ]:
total.to_csv('total_laggs.csv', index=False) # итоговые данные

In [46]:
total.shape

(3360, 24)

Теперь создадим объекты, но без подробной информации о предыдущих оценках: 

In [ ]:
total1 = df.copy()

total1['grade_10'] = pd.to_numeric(total1['grade_10'], errors='coerce')
total1['exam_type'] = pd.Categorical(total1['exam_type'], categories=['Первая сдача','Пересдача по уважительной причине','Пересдача','Пересдача с комиссией'], ordered=True)

total1 = total1.sort_values(['student_id', 'academic_year', 'module', 'exam_type', 'discipline']).reset_index(drop=True)

In [48]:
g_student = total1.groupby('student_id')
g_subject = total1.groupby(['student_id', 'discipline'])

total1['grade_prev'] = g_subject['grade_10'].shift(1)

total1['exam_type_prev'] = g_subject['exam_type'].shift(1)

total1['cnt_prev'] = g_student.cumcount()

total1['sum_prev'] = g_student['grade_10'].transform(lambda x: x.shift().cumsum())

total1['min_prev'] = g_student['grade_10'].transform(lambda x: x.shift().cummin())

total1['max_prev'] = g_student['grade_10'].transform(lambda x: x.shift().cummax())

total1['avg_grade_prev'] = total1['sum_prev'] / total1['cnt_prev']

total1 = total1[total1['cnt_prev'] > 0].copy()

In [49]:
total1['target_grade'] = g_subject['grade_10'].shift(-1)
total1['target_type'] = g_subject['exam_type'].shift(-1)
total1 = total1[~total1['target_grade'].isna()].copy()

In [50]:
total1 = total1.drop(columns=['cnt_prev', 'sum_prev'])

In [51]:
cols1 = [
    'student_id',
    'program', 'education_level', 'academic_year', 'place_type', 'course', 'absence_status', 'discipline',
    'module', 'exam_type', 'grade_10', 'difficulty_avg_score', 'std_deviation', 'students_responsed_ratio',
    'exam_type_prev','grade_prev',
    
    'avg_grade_prev',
    'min_prev','max_prev', 'target_grade', 'target_type'
    ]

total1 = total1.reindex(columns=cols1)

In [52]:
total1.to_csv('total_less_laggs.csv', index=False)

_Пример данных конкретного студента:_

In [53]:
total1[ total1['student_id'] == 240 ][['course', 'module', 'discipline', 'exam_type', 'grade_10', 'difficulty_avg_score', 'target_grade', 'target_type']]

,course,module,discipline,exam_type,grade_10,difficulty_avg_score,target_grade,target_type
6588,2,1,Дискретная математика,Первая сдача,6,NaN,7.0,Первая сдача
6591,2,2,Алгоритмы и структуры данных,Первая сдача,5,4.1579,6.0,Первая сдача
6592,2,2,Дискретная математика,Первая сдача,7,4.6111,8.0,Первая сдача
6593,2,2,История России,Первая сдача,8,4.3333,8.0,Первая сдача
6594,2,2,История России,Первая сдача,8,3.6667,8.0,Первая сдача
6595,2,2,Математический анализ 1,Первая сдача,5,NaN,6.0,Первая сдача
6596,2,2,Основы и методология программирования,Первая сдача,4,NaN,6.0,Первая сдача
6597,2,2,Язык программирования C++,Первая сдача,5,NaN,7.0,Первая сдача


In [54]:
total1.shape

(3360, 21)